In [26]:
%%writefile practice93.cpp
// Задание 3: Параллельный анализ графов (поиск кратчайших путей)
// 1. Процесс с "rank = 0" создаёт матрицу смежности графа G размером NxN.
// 2. Разделите строки матрицы между процессами с помощью функции "MPI_Scatter".
// 3. Реализуйте алгоритм Флойда-Уоршелла:
  // - Каждый процесс обновляет свою часть матрицы для текущей итерации.
  // - Передайте обновлённые данные между процессами с помощью функции "MPI_Allgather".
// 4. После завершения всех итераций соберите матрицу на процессе с "rank = 0" и выведите её на экран.

#include <mpi.h>          // Подключение библиотеки MPI для параллельного программирования
#include <iostream>       // Для вывода данных на экран
#include <vector>         // Для использования динамических массивов std::vector
#include <iomanip>        // Для форматирования вывода (setw)
#include <cstdlib>        // Для функций rand, atoi
#include <ctime>          // Для функции time (генератор случайных чисел)
#include <algorithm>      // Для стандартных алгоритмов (не используется напрямую, но может пригодиться)

const int INF = 1000000; // Большое число для обозначения отсутствия пути

int main(int argc, char* argv[]) {
    MPI_Init(&argc, &argv); // Инициализация MPI-среды

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank); // Получаем ранг текущего процесса (0, 1, 2, ...)
    MPI_Comm_size(MPI_COMM_WORLD, &size); // Получаем общее количество процессов в коммуникаторе

    int N = 8; // Размер графа по умолчанию (NxN)
    if (argc > 1) N = std::atoi(argv[1]); // Если пользователь передал параметр, используем его как размер графа

    std::vector<int> G_full; // Полная матрица смежности графа (только для rank 0)

    if (rank == 0) {                   // Процесс 0 создаёт исходную матрицу
        G_full.resize(N * N);          // Выделяем память под NxN элементов

        srand(42);                     // Фиксированный seed для одинаковой матрицы при разных запусках
        for (int i = 0; i < N; i++) {            // Цикл по строкам матрицы
            for (int j = 0; j < N; j++) {        // Цикл по столбцам матрицы
                if (i == j) G_full[i*N + j] = 0; // На диагонали = 0 (расстояние до самого себя)
                else {
                    int r = rand() % 10;                 // Случайное число 0..9
                    G_full[i*N + j] = (r < 7) ? (rand() % 10 + 1) : INF; // 70% вероятность ребра с весом 1..10, иначе INF
                }
            }
        }

        // Вывод исходной матрицы на экран
        std::cout << "Исходная матрица смежности:\n";
        for (int i = 0; i < N; i++) {                  // Цикл по строкам
            for (int j = 0; j < N; j++) {              // Цикл по столбцам
                if (G_full[i*N + j] >= INF)           // Если вес INF, выводим "INF"
                    std::cout << std::setw(5) << "INF" << " ";
                else
                    std::cout << std::setw(5) << G_full[i*N + j] << " "; // Иначе выводим число
            }
            std::cout << "\n";                        // Переход на новую строку после каждой строки матрицы
        }
        std::cout << "\n";                            // Пустая строка для разделения
    }

    // Определяем, сколько строк получает каждый процесс
    int rows_per_proc = N / size;             // Целое количество строк на процесс
    int remainder = N % size;                 // Остаток строк, если N не делится на size
    std::vector<int> sendcounts(size), displs(size); // Массивы для Scatterv: количество и смещение
    int offset = 0;
    for (int i = 0; i < size; i++) {
        sendcounts[i] = rows_per_proc + (i < remainder ? 1 : 0); // Первые remainder процессов получают на 1 строку больше
        displs[i] = offset;                                     // Смещение в массиве
        offset += sendcounts[i];                                 // Обновляем смещение
    }

    int local_rows = sendcounts[rank];             // Количество строк, обрабатываемых текущим процессом
    std::vector<int> local_G(local_rows * N);     // Локальная часть матрицы для текущего процесса

    // Настройка массивов sendcounts и displs для Scatterv (в элементах, а не строках)
    std::vector<int> sendcounts_G(size), displs_G(size);
    for (int i = 0; i < size; i++) {
        sendcounts_G[i] = sendcounts[i] * N;      // Количество элементов (строк * N) для каждого процесса
        displs_G[i] = displs[i] * N;              // Смещение в массиве
    }

    // Рассылаем строки матрицы между процессами
    MPI_Scatterv(G_full.data(), sendcounts_G.data(), displs_G.data(), MPI_INT,
                 local_G.data(), local_rows * N, MPI_INT, 0, MPI_COMM_WORLD);

    double start_time = MPI_Wtime(); // Начало измерения времени выполнения

    std::vector<int> k_row(N);       // Вектор для хранения k-й строки (строка, которая рассылается)

    // Основной цикл алгоритма Флойда-Уоршелла
    for (int k = 0; k < N; k++) {
        int owner = 0;
        while (k >= displs[owner] + sendcounts[owner]) owner++; // Определяем, какой процесс владеет строкой k

        if (rank == owner) {                   // Если текущий процесс владеет k-й строкой
            int local_index = k - displs[rank];          // Локальный индекс строки
            for (int j = 0; j < N; j++)
                k_row[j] = local_G[local_index * N + j]; // Копируем k-ю строку в отдельный массив
        }

        // Рассылаем k-ю строку всем процессам
        MPI_Bcast(k_row.data(), N, MPI_INT, owner, MPI_COMM_WORLD);

        // Обновляем локальные строки
        for (int i_local = 0; i_local < local_rows; i_local++) {  // По строкам локального блока
            for (int j = 0; j < N; j++) {                          // По столбцам
                // Проверяем, можно ли сократить путь через k
                if (local_G[i_local * N + k] + k_row[j] < local_G[i_local * N + j])
                    local_G[i_local * N + j] = local_G[i_local * N + k] + k_row[j]; // Обновляем минимальное расстояние
            }
        }
    }

    // Собираем локальные блоки обратно в полную матрицу на rank 0
    MPI_Gatherv(local_G.data(), local_rows * N, MPI_INT,
                G_full.data(), sendcounts_G.data(), displs_G.data(), MPI_INT,
                0, MPI_COMM_WORLD);

    double end_time = MPI_Wtime(); // Конец измерения времени

    // Вывод на экран (только процесс 0)
    if (rank == 0) {
        std::cout << "Матрица кратчайших путей:\n";
        for (int i = 0; i < N; i++) {                  // По строкам
            for (int j = 0; j < N; j++) {              // По столбцам
                if (G_full[i*N + j] >= INF)
                    std::cout << std::setw(5) << "INF" << " "; // Вывод INF для недостижимых вершин
                else
                    std::cout << std::setw(5) << G_full[i*N + j] << " "; // Вывод минимального расстояния
            }
            std::cout << "\n";
        }
        std::cout << "Время выполнения = " << end_time - start_time << " с.\n"; // Время выполнения
    }

    MPI_Finalize(); // Завершение MPI-среды
    return 0;       // Завершение программы
}


Overwriting practice93.cpp


In [28]:
# Компиляция
!mpic++ practice93.cpp -o practice93
# mpirun — утилита, которая запускает MPI-программу на указанном числе процессов
# --allow-run-as-root — разрешает запуск MPI от root (Colab запускает ядра как root)
# --oversubscribe — игнорирует количество доступных виртуальных CPU, позволяя запускать больше процессов, чем физически есть
# -np 2 — число процессов MPI (2 процесса)

# Запуск с 2 процессами
!mpirun --allow-run-as-root --oversubscribe -np 2 ./practice93 5
# Запуск с 8 процессами
!mpirun --allow-run-as-root --oversubscribe -np 8 ./practice93 5
# Запуск с 16 процессами
!mpirun --allow-run-as-root --oversubscribe -np 16 ./practice93 5

Исходная матрица смежности:
    0     1     2     9     1 
    4     0     4   INF     7 
    3   INF     0   INF   INF 
  INF    10     8     0   INF 
  INF     2     4   INF     0 

Матрица кратчайших путей:
    0     1     2     9     1 
    4     0     4    13     5 
    3     4     0    12     4 
   11    10     8     0    12 
    6     2     4    15     0 
Время выполнения = 0.000206847 с.
Исходная матрица смежности:
    0     1     2     9     1 
    4     0     4   INF     7 
    3   INF     0   INF   INF 
  INF    10     8     0   INF 
  INF     2     4   INF     0 

Матрица кратчайших путей:
    0     1     2     9     1 
    4     0     4    13     5 
    3     4     0    12     4 
   11    10     8     0    12 
    6     2     4    15     0 
Время выполнения = 0.000667111 с.
Исходная матрица смежности:
    0     1     2     9     1 
    4     0     4   INF     7 
    3   INF     0   INF   INF 
  INF    10     8     0   INF 
  INF     2     4   INF     0 

Матрица кратчайших